In [1]:
import pandas as pd
import numpy as np
import json
from collections import Counter
from sklearn.metrics import f1_score

In [2]:
df = pd.read_csv("../data/CF/cf_v1_test.csv")

out_files = ["zero_shot_qwen3-14b_False_False"]

out_files = ["zero_shot_aya-expanse-8b_False_False",
             "zero_shot_ministral-8b_False_False",
             "zero_shot_eurollm-9b_False_False",
             "zero_shot_eurollm-22b_False_False",
             "zero_shot_aya-expanse-32b_False_False",
             "zero_shot_qwen3-14b_False_False",
             "zero_shot_qwen3-14b_True_False",
             "zero_shot_qwen3-32b_False_False",
             "zero_shot_qwen3-32b_True_False",
             "zero_shot_gpt-5-nano_False_False",
             "zero_shot_gpt-5-nano_True_False",
             "web_search_gpt-5-nano_True_True",
             "web_search_URL_gpt-5-nano_True_True",
             "zero_shot_gemini-2.5-flash_False_False",
             "zero_shot_gemini-2.5-flash_True_False",
             "web_search_gemini-2.5-flash_True_True",
             "web_search_URL_gemini-2.5-flash_True_True",
             "zero_shot_grok-4.3_False_False",
             "zero_shot_grok-4.3_True_False",
             "web_search_grok-4.3_True_True",
             "web_search_URL_grok-4.3_True_True",
             ]

scores = [] # pd.DataFrame(columns = [f"{d}_{l}" for d in ["politics", "finance"] for l in ["en", "es", "fr", "ja", "pt"]])
for outf in out_files:
    dic = json.load(open(f"outputs/{outf}.json", "r"))
    df["pred"] = [True if "true" in i.lower() else (False if "false" in i.lower() else None) for i in dic.values()] # list(dic.values())
    df = df[df.pred != None]
    df["pred"] = df["pred"].astype(bool)
    tmp_dic = {}
    for d in ["politics", "finance"]:
        for l in ["en", "es", "fr", "ja", "pt"]:
            tmp = df[(df.language==l) & (df.domain==d)]
            tmp_dic[f"{d}_{l}"] = np.round(f1_score(tmp.label, tmp.pred, average="macro")*100, 2)
    tmp_dic["overall"] = np.round(f1_score(df.label, df.pred, average="macro")*100, 2)
            
    scores.append(tmp_dic)

scores = pd.DataFrame(scores)
scores.insert(0, "method", out_files)
scores.to_csv("scores.csv")
scores

,method,politics_en,politics_es,politics_fr,politics_ja,politics_pt,finance_en,finance_es,finance_fr,finance_ja,finance_pt,overall
0,zero_shot_aya-expanse-8b_False_False,62.71,61.10,61.67,69.63,62.47,59.35,65.21,65.52,74.97,65.50,63.74
1,zero_shot_ministral-8b_False_False,39.59,36.79,37.28,35.04,34.56,37.54,36.92,31.49,34.44,38.64,36.74
2,zero_shot_eurollm-9b_False_False,47.06,40.60,45.57,52.43,47.72,45.12,43.19,39.06,48.22,49.44,46.46
3,zero_shot_eurollm-22b_False_False,54.75,56.50,56.17,71.21,59.53,51.94,56.09,64.15,66.88,60.36,59.23
4,zero_shot_aya-expanse-32b_False_False,71.79,69.37,66.04,74.90,66.71,69.70,69.89,59.72,79.13,70.95,69.93
5,zero_shot_qwen3-14b_False_False,62.07,62.07,59.80,70.40,63.72,59.60,61.94,53.77,64.53,63.14,62.83
6,zero_shot_qwen3-14b_True_False,56.99,54.80,56.39,67.00,57.90,57.79,59.26,56.46,62.84,62.31,58.92
7,zero_shot_qwen3-32b_False_False,68.63,65.15,62.70,66.80,64.83,64.40,65.73,62.21,73.03,65.36,65.68
8,zero_shot_qwen3-32b_True_False,62.65,57.75,60.56,62.19,60.72,61.09,60.18,60.38,66.11,62.71,61.15
9,zero_shot_gpt-5-nano_False_False,39.45,36.79,37.01,38.08,37.39,38.16,34.48,35.01,40.35,43.21,38.09


In [3]:
def compare_f1_significance(
    y_true,
    y_pred_model_1,
    y_pred_model_2,
    average="macro",
    n_permutations=10000,
    n_bootstrap=10000,
    random_state=42,
    zero_division=0,
    confidence_level=0.95,
):
    """
    Compare two models on small language/domain subsets.

    Uses:
    1. Paired permutation test for p-value.
    2. Stratified paired bootstrap for confidence interval.

    Recommended for n around 100-400, especially with macro-F1.
    """

    y_true = np.asarray(y_true)
    y1 = np.asarray(y_pred_model_1)
    y2 = np.asarray(y_pred_model_2)

    if not (len(y_true) == len(y1) == len(y2)):
        raise ValueError("All inputs must have the same length.")

    if len(y_true) == 0:
        raise ValueError("Inputs must not be empty.")

    rng = np.random.default_rng(random_state)
    n = len(y_true)

    labels = np.unique(y_true)
    label_counts = Counter(y_true)

    warnings = []

    if n < 100:
        warnings.append(
            "Very small subset. Significance results may be unstable."
        )

    small_classes = {
        str(label): int(count)
        for label, count in label_counts.items()
        if count < 5
    }

    if average == "macro" and small_classes:
        warnings.append(
            f"Some classes have fewer than 5 examples: {small_classes}. "
            "Macro-F1 may be unstable."
        )

    def score(yt, yp):
        return f1_score(
            yt,
            yp,
            average=average,
            labels=labels,
            zero_division=zero_division,
        )

    # Original scores
    f1_1 = score(y_true, y1)
    f1_2 = score(y_true, y2)
    observed_diff = f1_1 - f1_2

    # --------------------------------------------------
    # 1. Paired permutation test for p-value
    # --------------------------------------------------
    extreme_count = 0

    for _ in range(n_permutations):
        swap = rng.random(n) < 0.5

        perm_y1 = y1.copy()
        perm_y2 = y2.copy()

        perm_y1[swap] = y2[swap]
        perm_y2[swap] = y1[swap]

        perm_diff = score(y_true, perm_y1) - score(y_true, perm_y2)

        if abs(perm_diff) >= abs(observed_diff):
            extreme_count += 1

    p_value = (extreme_count + 1) / (n_permutations + 1)

    # --------------------------------------------------
    # 2. Stratified paired bootstrap for confidence interval
    # --------------------------------------------------
    class_indices = [np.where(y_true == label)[0] for label in labels]

    bootstrap_diffs = []

    for _ in range(n_bootstrap):
        sampled_indices = []

        for idx in class_indices:
            sampled_idx = rng.choice(idx, size=len(idx), replace=True)
            sampled_indices.append(sampled_idx)

        sampled_indices = np.concatenate(sampled_indices)

        yt = y_true[sampled_indices]
        yp1 = y1[sampled_indices]
        yp2 = y2[sampled_indices]

        diff = score(yt, yp1) - score(yt, yp2)
        bootstrap_diffs.append(diff)

    bootstrap_diffs = np.asarray(bootstrap_diffs)

    alpha = 1.0 - confidence_level

    ci_low, ci_high = np.percentile(
        bootstrap_diffs,
        [100 * alpha / 2, 100 * (1 - alpha / 2)]
    )

    # Significance stars
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    else:
        sig = "ns"

    return {
        "f1_model_1": round(float(f1_1), 6),
        "f1_model_2": round(float(f1_2), 6),
        "observed_diff": round(float(observed_diff), 6),
        "p_value": round(float(p_value), 6),
        "significance": sig,
        "ci_low": round(float(ci_low), 6),
        "ci_high": round(float(ci_high), 6),
        "ci_contains_zero": bool(ci_low <= 0 <= ci_high),
        "n_samples": int(n),
        "label_counts": {str(k): int(v) for k, v in label_counts.items()},
        "warnings": warnings,
    }

In [4]:
out_files = [["web_search_URL_gpt-5-nano_True_True", "web_search_gpt-5-nano_True_True"],
             ["web_search_URL_gemini-2.5-flash_True_True", "web_search_gemini-2.5-flash_True_True"],
             ["web_search_URL_grok-4.3_True_True", "web_search_grok-4.3_True_True"]
]

for outf1, outf2 in out_files:
    df = pd.read_csv("../data/CF/cf_v1_test.csv")
    dic1 = json.load(open(f"outputs/{outf1}.json", "r"))
    dic2 = json.load(open(f"outputs/{outf2}.json", "r"))
    df["pred1"] = [True if "true" in i.lower() else (False if "false" in i.lower() else None) for i in dic1.values()]
    df["pred2"] = [True if "true" in i.lower() else (False if "false" in i.lower() else None) for i in dic2.values()]
    df = df[df.pred1 != None]
    df = df[df.pred2 != None]
    df["pred1"] = df["pred1"].astype(bool)
    df["pred2"] = df["pred2"].astype(bool)

    print(f"Comparing {outf1} and {outf2}...\n")
    # Note: The subgroup analysis is commented out for now, as the sample sizes in some subgroups are very small, leading to unstable significance results. The overall comparison is more reliable.
    # for d in ["politics", "finance"]:
    #     for l in ["en", "es", "fr", "ja", "pt"]:
    #         tmp = df[(df.language==l) & (df.domain==d)]
    #         results = compare_f1_significance(
    #             y_true=tmp.label,
    #             y_pred_model_1=tmp.pred1,
    #             y_pred_model_2=tmp.pred2
    #         )
    #         print(f"{d} --- {l}:", results)
    print("\nOverall:", compare_f1_significance(
        y_true=df.label,
        y_pred_model_1=df.pred1,
        y_pred_model_2=df.pred2
    ), end="\n\n")

Comparing web_search_URL_gpt-5-nano_True_True and web_search_gpt-5-nano_True_True...


Overall: {'f1_model_1': 0.770686, 'f1_model_2': 0.755942, 'observed_diff': 0.014744, 'p_value': 0.021598, 'significance': '*', 'ci_low': 0.002166, 'ci_high': 0.027394, 'ci_contains_zero': False, 'n_samples': 3578, 'label_counts': {'True': 2246, 'False': 1332}, 'warnings': []}

Comparing web_search_URL_gemini-2.5-flash_True_True and web_search_gemini-2.5-flash_True_True...


Overall: {'f1_model_1': 0.77465, 'f1_model_2': 0.781292, 'observed_diff': -0.006643, 'p_value': 0.315768, 'significance': 'ns', 'ci_low': -0.019228, 'ci_high': 0.006005, 'ci_contains_zero': True, 'n_samples': 3578, 'label_counts': {'True': 2246, 'False': 1332}, 'warnings': []}

Comparing web_search_URL_grok-4.3_True_True and web_search_grok-4.3_True_True...


Overall: {'f1_model_1': 0.857693, 'f1_model_2': 0.837998, 'observed_diff': 0.019695, 'p_value': 0.0001, 'significance': '***', 'ci_low': 0.009441, 'ci_high': 0.029912, 'ci_co